# medeval Demo

PyTorch-native evaluation metrics for medical imaging: 2D/3D segmentation, classification, detection, and registration with physical spacing support, bootstrap CI, and visualization.


In [ ]:
# Install dependencies (run once)
# - medeval[all] pulls in torch, IO (nibabel/SimpleITK), CLI deps (pandas/tqdm/pyyaml), and matplotlib
%pip install -U "medeval[all]"

In [ ]:
import numpy as np
import torch
import medeval
import matplotlib

print(f"medeval: {medeval.__version__}, torch: {torch.__version__}, matplotlib: {matplotlib.__version__}, device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Segmentation Metrics

Overlap (Dice, Jaccard, Precision, Recall, Volumetric Similarity) and surface distances (Hausdorff, HD95, ASSD) with physical spacing support.


In [ ]:
from medeval.metrics.segmentation import compute_segmentation_metrics

# 3D binary segmentation: shape (B, C, D, H, W), anisotropic spacing (dz, dy, dx) in mm
torch.manual_seed(0)
pred = (torch.rand(1, 1, 32, 64, 64) > 0.5).int()
target = (torch.rand(1, 1, 32, 64, 64) > 0.5).int()
spacing = (3.0, 1.0, 1.0)  # 3mm slice thickness, 1mm in-plane

# Basic overlap metrics
results = compute_segmentation_metrics(pred, target, spacing=spacing, include_surface=False)
print("Overlap metrics:")
for k, v in results.items():
    print(f"  {k}: {v.item():.4f}" if hasattr(v, 'item') else f"  {k}: {v:.4f}")

In [ ]:
# Surface distance metrics (use smaller volume for speed)
pred_small = torch.zeros(1, 1, 10, 20, 20)
target_small = torch.zeros(1, 1, 10, 20, 20)
pred_small[0, 0, 2:6, 5:15, 5:15] = 1  # cube
target_small[0, 0, 3:7, 6:16, 6:16] = 1  # shifted cube

results_surface = compute_segmentation_metrics(pred_small, target_small, spacing=(3.0, 1.0, 1.0), include_surface=True)
print("Surface distances (mm):")
for k, v in results_surface.items():
    unit = " mm" if "hausdorff" in k.lower() or "assd" in k.lower() else ""
    print(f"  {k}: {v.item():.4f}{unit}" if hasattr(v, 'item') else f"  {k}: {v:.4f}{unit}")

In [ ]:
# Multi-class segmentation with integer labels (class indices 0, 1, 2)
# Uses the fixed multi-class Dice implementation that properly handles integer labels
from medeval.metrics.segmentation import dice_score

torch.manual_seed(42)
pred_multi = torch.randint(0, 3, (4, 1, 16, 32, 32))  # 4 cases, 3 classes
target_multi = torch.randint(0, 3, (4, 1, 16, 32, 32))

# Test different reduction strategies
print("Multi-class segmentation with integer labels:")
for reduction in ['none', 'mean-case', 'global']:
    d = dice_score(pred_multi, target_multi, reduction=reduction)
    if hasattr(d, 'shape') and d.numel() > 1:
        # Per-case results
        print(f"  reduction='{reduction}': dice shape={d.shape}, mean={d.mean().item():.4f}")
    else:
        val = d.item() if hasattr(d, 'item') else d
        print(f"  reduction='{reduction}': dice={val:.4f}")


## 2. Classification Metrics

AUROC, AUPRC, accuracy, F1, MCC, Cohen's kappa, calibration (ECE, Brier), with bootstrap confidence intervals.


In [ ]:
from medeval.metrics.classification import compute_classification_metrics

np.random.seed(0)
probs = np.random.rand(200)
labels = (np.random.rand(200) > 0.75).astype(int)  # ~25% positive

# Basic classification metrics
cls_results = compute_classification_metrics(probs, labels)
print("Classification metrics:")
for k, v in cls_results.items():
    if isinstance(v, tuple):  # CI tuple
        print(f"  {k}: {v[0]:.4f} [{v[1]:.4f}, {v[2]:.4f}]")
    elif hasattr(v, 'item'):
        print(f"  {k}: {v.item():.4f}")
    else:
        print(f"  {k}: {v:.4f}")

In [ ]:
# With bootstrap 95% CI and calibration metrics
cls_with_ci = compute_classification_metrics(
    probs, labels, 
    compute_ci=True, 
    confidence=0.95, 
    include_calibration=True
)
print("\nWith CI and calibration:")
for k, v in cls_with_ci.items():
    if isinstance(v, tuple) and len(v) == 3:
        print(f"  {k}: {v[0]:.4f} [95% CI: {v[1]:.4f}, {v[2]:.4f}]")
    elif hasattr(v, 'item'):
        print(f"  {k}: {v.item():.4f}")
    else:
        print(f"  {k}: {v:.4f}")

## 3. Detection Metrics

IoU, mAP, FROC for 2D/3D bounding box and instance segmentation.

In [ ]:
from medeval.metrics.detection import compute_detection_metrics, box_iou_3d

# 3D bounding boxes: [x1, y1, z1, x2, y2, z2, score, class_id]
pred_boxes = torch.tensor([
    [10, 10, 5, 30, 30, 15, 0.9, 0],   # High confidence class 0 (matches GT)
    [50, 50, 10, 70, 70, 20, 0.8, 0],  # Medium confidence class 0 (FP - no matching GT)
    [80, 80, 5, 100, 100, 15, 0.7, 1], # Class 1 (matches GT)
], dtype=torch.float32)

gt_boxes = torch.tensor([
    [12, 12, 6, 32, 32, 16, 1.0, 0],   # Ground truth class 0
    [82, 82, 6, 102, 102, 16, 1.0, 1], # Ground truth class 1
], dtype=torch.float32)

det_results = compute_detection_metrics(
    pred_boxes, gt_boxes, 
    iou_thresholds=[0.5, 0.75], 
    use_3d=True,
    include_froc=True
)

print("Detection metrics:")
for k, v in det_results.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: array of length {len(v)}")
    elif hasattr(v, 'item'):
        print(f"  {k}: {v.item():.4f}")
    else:
        print(f"  {k}: {v:.4f}")


## 4. Registration Metrics

Landmark TRE, image similarity (NMI, NCC), and deformation field quality (Jacobian determinant).


In [ ]:
from medeval.metrics.registration import compute_registration_metrics, normalized_mutual_information

# Landmark-based: predicted vs ground truth landmark positions (N, 3) in mm
pred_landmarks = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0], [70.0, 80.0, 90.0]])
gt_landmarks = torch.tensor([[10.5, 20.2, 30.1], [40.8, 50.5, 60.3], [71.0, 81.0, 91.0]])

# Deformation field: (3, Z, H, W) displacement vectors
torch.manual_seed(0)
deformation = torch.randn(3, 16, 32, 32) * 0.1

reg_results = compute_registration_metrics(
    pred_landmarks=pred_landmarks, 
    target_landmarks=gt_landmarks,
    deformation_field=deformation,
    spacing=(2.0, 1.0, 1.0)
)

print("Registration metrics:")
for k, v in reg_results.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for k2, v2 in v.items():
            if isinstance(v2, np.ndarray):
                print(f"    {k2}: array of shape {v2.shape}")
            elif isinstance(v2, float):
                print(f"    {k2}: {v2:.4f}")
            else:
                print(f"    {k2}: {v2}")
    elif isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


## 5. Visualization

ROC/PR curves, calibration plots, and error distributions.


In [ ]:
import medeval.vis as vis
import matplotlib.pyplot as plt

# Check what's available
available = [f for f in dir(vis) if not f.startswith('_') and callable(getattr(vis, f, None))]
print(f"Available vis functions: {available}")

# Generate sample data for visualization
np.random.seed(42)
y_true = np.random.randint(0, 2, 200)
# Create scores that are correlated with labels for realistic plots
y_score = np.clip(y_true * 0.6 + np.random.randn(200) * 0.3, 0, 1)

# ROC and PR curves
# Use plot_roc_from_predictions / plot_pr_from_predictions for raw y_true/y_score
# These functions compute the curve points internally using sklearn
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vis.plot_roc_from_predictions(y_true, y_score, ax=axes[0])
axes[0].set_title('ROC Curve')

vis.plot_pr_from_predictions(y_true, y_score, ax=axes[1])
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()


In [ ]:
# Calibration and reliability diagrams
# Use plot_reliability_from_predictions for convenience - computes calibration internally
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Reliability diagram with ECE annotation and gap visualization
# n_bins=10 is standard; the function computes and displays ECE automatically
vis.plot_reliability_from_predictions(y_true, y_score, n_bins=10, ax=axes[0])
axes[0].set_title('Reliability Diagram (10 uniform bins)')

vis.plot_confidence_histogram(y_score, labels=y_true, ax=axes[1])
axes[1].set_title('Confidence Histogram')

plt.tight_layout()
plt.show()

In [ ]:
# Error histogram for segmentation metrics
# Simulating typical Dice scores: most cases good (0.7-0.9), some failure cases in tail
dice_scores = np.random.beta(8, 2, 100)  # Left-skewed distribution typical for Dice

fig, ax = plt.subplots(figsize=(8, 5))
vis.plot_error_histogram(
    dice_scores, 
    bins=15,  # Fewer bins for cleaner visualization
    xlabel='Dice Score', 
    title='Dice Score Distribution',
    ax=ax
)
# Legend auto-positions to upper left (opposite of data mass)
# Stats box shows n, mean, std
plt.show()


## 6. Aggregation & Stratification

Bootstrap CI and stratified aggregation by site/scanner.


In [ ]:
try:
    from medeval.core.aggregate import bootstrap_ci, stratified_aggregate

    # Per-case metrics from 20 samples across 2 sites
    np.random.seed(0)
    dice_scores = np.random.rand(20) * 0.3 + 0.6  # Dice scores 0.6-0.9
    site_ids = np.array([0] * 10 + [1] * 10)  # Use integer IDs

    # Bootstrap 95% CI
    mean, ci_low, ci_high = bootstrap_ci(dice_scores, confidence=0.95, n_bootstrap=1000)
    print(f"Dice: {mean:.4f} [95% CI: {ci_low:.4f}, {ci_high:.4f}]")

    # Stratified by site (metrics must be a dict)
    # Use .tolist() to keep the example friendly to static type checkers.
    metrics_dict = {"dice": dice_scores.tolist()}
    strat_results = stratified_aggregate(metrics_dict, site_ids, compute_ci=True)

    print("\nStratified by site:")
    for metric_name, per_site in strat_results.items():
        print(f"  {metric_name}:")
        for site, val in per_site.items():
            if isinstance(val, tuple) and len(val) == 3:
                m, lo, hi = val
                print(f"    site={site}: {m:.4f} [95% CI: {lo:.4f}, {hi:.4f}]")
            else:
                print(f"    site={site}: {val}")
except ImportError:
    print("Aggregate module not available")


## 7. CLI

Batch evaluation from command line with YAML/JSON config.


In [ ]:
# CLI usage example (run in terminal)
print("""CLI examples:
  medeval segmentation --pred pred.nii.gz --target target.nii.gz --spacing 1,1,3
  medeval classification --pred probs.csv --target labels.csv --ci
  medeval detection --pred boxes.json --target gt.json --iou 0.5
  medeval --config eval_config.yaml  # Batch evaluation from config file
""")


## 8. IO & Interoperability

Supports NumPy, PyTorch, NIfTI (SimpleITK), and optional MONAI/torchmetrics interop.


In [ ]:
from medeval.core.io import load_image, get_spacing_from_header, get_nifti_spacing
from medeval.core.typing import as_tensor, to_device

# Convert between formats
arr_np = np.random.rand(32, 64, 64)
arr_torch = as_tensor(arr_np)  # NumPy -> PyTorch
print(f"NumPy {arr_np.shape} -> PyTorch {arr_torch.shape}")

# GPU support (if available)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
arr_gpu = as_tensor(arr_np, device=device)
print(f"Device: {arr_gpu.device}")

# Spacing utilities info
print("\nSpacing utilities available:")
print("  - get_spacing_from_header(path): Auto-detect format and extract spacing")
print("  - get_nifti_spacing(path): Extract from NIfTI files")
print("  - get_sitk_spacing(path): Extract from SimpleITK-readable files")
print("  - get_dicom_spacing(path): Extract from DICOM files")


## 9. End-to-End Demo (Self-contained)

This section is **fully self-contained**:

- It generates **synthetic (no-PHI)** data directly inside this notebook
- It writes temporary NIfTI/NPY files + CSV manifests + YAML configs
- It runs the CLI (`python -m medeval.cli.main evaluate ...`) for **segmentation / classification / detection / registration**
- It reads back `summary.json` + `results.csv` and visualizes the outputs

Everything is written into a local folder: `./medeval_demo_artifacts/`


In [ ]:
import csv
import json
import sys
import subprocess
from pathlib import Path

import numpy as np
import yaml

from medeval.core.io import save_nifti

ARTIFACTS_DIR = Path.cwd() / "medeval_demo_artifacts"
DATA_DIR = ARTIFACTS_DIR / "data"
SEG_DIR = DATA_DIR / "seg"
CLS_DIR = DATA_DIR / "cls"
REG_DIR = DATA_DIR / "reg"
DET_DIR = DATA_DIR / "det"

OUT_SEG = ARTIFACTS_DIR / "out_cli_seg"
OUT_CLS = ARTIFACTS_DIR / "out_cli_cls"
OUT_REG = ARTIFACTS_DIR / "out_cli_reg"
OUT_DET = ARTIFACTS_DIR / "out_cli_det"

for d in [SEG_DIR, CLS_DIR, REG_DIR, DET_DIR, OUT_SEG, OUT_CLS, OUT_REG, OUT_DET]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifacts dir: {ARTIFACTS_DIR}")

# -------------------------
# 1) Synthetic SEGMENTATION dataset (3D NIfTI, anisotropic spacing)
# -------------------------

def _sphere_mask(shape_zyx, center_zyx, radius_vox):
    z, y, x = shape_zyx
    zz, yy, xx = np.ogrid[:z, :y, :x]
    cz, cy, cx = center_zyx
    dist2 = (zz - cz) ** 2 + (yy - cy) ** 2 + (xx - cx) ** 2
    return (dist2 <= radius_vox**2).astype(np.float32)

rng = np.random.default_rng(123)
shape = (32, 96, 96)  # (Z,Y,X)
spacing = (3.0, 1.0, 1.0)  # (dz,dy,dx)

rows = []
n_patients = 10
cases_per_patient = 2
strata = ["site_A", "site_B"]

for p in range(n_patients):
    patient_id = f"patient_{p+1:03d}"
    site = strata[p % len(strata)]
    for k in range(cases_per_patient):
        case_id = f"case_{p:02d}_{k:02d}"

        center_gt = (16, 48, 48)
        shift = (int(rng.integers(-1, 2)), int(rng.integers(-3, 4)), int(rng.integers(-3, 4)))
        center_pred = (center_gt[0] + shift[0], center_gt[1] + shift[1], center_gt[2] + shift[2])

        target = _sphere_mask(shape, center_gt, radius_vox=12)
        pred = _sphere_mask(shape, center_pred, radius_vox=12)

        pred_path = SEG_DIR / f"{case_id}_pred.nii.gz"
        tgt_path = SEG_DIR / f"{case_id}_tgt.nii.gz"
        save_nifti(pred, str(pred_path), spacing=spacing)
        save_nifti(target, str(tgt_path), spacing=spacing)

        rows.append(
            {
                "prediction": str(pred_path),
                "target": str(tgt_path),
                "spacing": ",".join(map(str, spacing)),
                "patient_id": patient_id,
                "strata": site,
            }
        )

seg_manifest = ARTIFACTS_DIR / "manifest_seg.csv"
with open(seg_manifest, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["prediction", "target", "spacing", "patient_id", "strata"])
    w.writeheader()
    w.writerows(rows)

seg_config = ARTIFACTS_DIR / "config_seg.yaml"
seg_config.write_text(
    yaml.safe_dump(
        {
            "task": "segmentation",
            "columns": {
                "prediction": "prediction",
                "target": "target",
                "spacing": "spacing",
                "patient_id": "patient_id",
                "strata": "strata",
            },
            "metrics": {"threshold": 0.5, "include_surface": True, "include_calibration": False},
            "aggregation": {"confidence": 0.95, "n_bootstrap": 200, "seed": 42},
        }
    )
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "medeval.cli.main",
        "evaluate",
        "--task",
        "segmentation",
        "--manifest",
        str(seg_manifest),
        "--config",
        str(seg_config),
        "--output",
        str(OUT_SEG),
    ],
    check=True,
)

# -------------------------
# 2) Synthetic CLASSIFICATION dataset (.npy arrays)
# -------------------------

n = 500
labels = (rng.random(n) > 0.75).astype(np.int64)
# create signal: positives higher
probs = rng.beta(2, 5, size=n).astype(np.float32)
probs[labels == 1] = rng.beta(5, 2, size=int((labels == 1).sum())).astype(np.float32)
probs = np.clip(probs + rng.normal(0, 0.08, size=n), 0.01, 0.99).astype(np.float32)

probs_path = CLS_DIR / "probs.npy"
labels_path = CLS_DIR / "labels.npy"
np.save(probs_path, probs)
np.save(labels_path, labels)

cls_manifest = ARTIFACTS_DIR / "manifest_cls.csv"
with open(cls_manifest, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["prediction", "target", "strata"])
    w.writeheader()
    w.writerow({"prediction": str(probs_path), "target": str(labels_path), "strata": "site_A"})

cls_config = ARTIFACTS_DIR / "config_cls.yaml"
cls_config.write_text(
    yaml.safe_dump(
        {
            "task": "classification",
            "columns": {"prediction": "prediction", "target": "target", "strata": "strata"},
            "metrics": {"include_calibration": True},
            "aggregation": {"confidence": 0.95, "n_bootstrap": 500, "seed": 42},
        }
    )
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "medeval.cli.main",
        "evaluate",
        "--task",
        "classification",
        "--manifest",
        str(cls_manifest),
        "--config",
        str(cls_config),
        "--output",
        str(OUT_CLS),
    ],
    check=True,
)

# -------------------------
# 3) Synthetic REGISTRATION dataset (.npy images)
# -------------------------

def _gaussian_2d(h, w, cy, cx, sigma=6.0):
    yy, xx = np.mgrid[:h, :w]
    g = np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma**2)))
    return g.astype(np.float32)

img_t = _gaussian_2d(64, 64, 32, 32)
img_p = _gaussian_2d(64, 64, 32, 34)  # slight shift

pred_img_path = REG_DIR / "pred.npy"
tgt_img_path = REG_DIR / "tgt.npy"
np.save(pred_img_path, img_p)
np.save(tgt_img_path, img_t)

reg_manifest = ARTIFACTS_DIR / "manifest_reg.csv"
with open(reg_manifest, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["prediction", "target", "spacing"])
    w.writeheader()
    w.writerow({"prediction": str(pred_img_path), "target": str(tgt_img_path), "spacing": "1.0,1.0"})

reg_config = ARTIFACTS_DIR / "config_reg.yaml"
reg_config.write_text(
    yaml.safe_dump(
        {
            "task": "registration",
            "columns": {"prediction": "prediction", "target": "target", "spacing": "spacing"},
            "metrics": {"include_image_similarity": True},
            "aggregation": {"confidence": 0.95, "n_bootstrap": 50, "seed": 42},
        }
    )
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "medeval.cli.main",
        "evaluate",
        "--task",
        "registration",
        "--manifest",
        str(reg_manifest),
        "--config",
        str(reg_config),
        "--output",
        str(OUT_REG),
    ],
    check=True,
)

# -------------------------
# 4) Synthetic DETECTION dataset (.npy box tables)
# -------------------------

pred_boxes = np.array(
    [
        [10, 10, 5, 30, 30, 15, 0.9, 0],
        [50, 50, 10, 70, 70, 20, 0.8, 0],
        [80, 80, 5, 100, 100, 15, 0.7, 1],
    ],
    dtype=np.float32,
)
gt_boxes = np.array(
    [
        [12, 12, 6, 32, 32, 16, 1.0, 0],
        [82, 82, 6, 102, 102, 16, 1.0, 1],
    ],
    dtype=np.float32,
)

pred_box_path = DET_DIR / "pred_boxes.npy"
gt_box_path = DET_DIR / "gt_boxes.npy"
np.save(pred_box_path, pred_boxes)
np.save(gt_box_path, gt_boxes)

det_manifest = ARTIFACTS_DIR / "manifest_det.csv"
with open(det_manifest, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["prediction", "target"])
    w.writeheader()
    w.writerow({"prediction": str(pred_box_path), "target": str(gt_box_path)})

det_config = ARTIFACTS_DIR / "config_det.yaml"
det_config.write_text(
    yaml.safe_dump(
        {
            "task": "detection",
            "columns": {"prediction": "prediction", "target": "target"},
            "metrics": {"use_3d": True, "iou_thresholds": [0.5, 0.75], "include_froc": False},
            "aggregation": {"confidence": 0.95, "n_bootstrap": 50, "seed": 42},
        }
    )
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "medeval.cli.main",
        "evaluate",
        "--task",
        "detection",
        "--manifest",
        str(det_manifest),
        "--config",
        str(det_config),
        "--output",
        str(OUT_DET),
    ],
    check=True,
)

print("\nDone. CLI outputs:")
print(f"  - Segmentation:  {OUT_SEG}")
print(f"  - Classification: {OUT_CLS}")
print(f"  - Registration:   {OUT_REG}")
print(f"  - Detection:      {OUT_DET}")



In [ ]:
import json
import sys
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Ensure NIfTI support is available (nibabel)
try:
    import nibabel as _  # noqa: F401
except Exception:
    print("nibabel not found; installing it now...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "nibabel"], check=True)

from medeval.core.io import get_nifti_spacing

ARTIFACTS_DIR = Path.cwd() / "medeval_demo_artifacts"

# ---------
# Segmentation CLI outputs
# ---------
seg_summary_path = ARTIFACTS_DIR / "out_cli_seg" / "summary.json"
seg_summary = json.loads(seg_summary_path.read_text())
print(f"SEG CLI processed: {seg_summary['n_processed']}/{seg_summary['n_samples']} (errors={seg_summary['n_errors']})")
print("Key SEG metrics (mean [95% CI]):")
for k in ["dice", "hausdorff_95", "assd"]:
    m = seg_summary["metrics"][k]["mean"]
    lo = seg_summary["metrics"][k]["ci_lower"]
    hi = seg_summary["metrics"][k]["ci_upper"]
    print(f"  {k}: {m:.4f} [{lo:.4f}, {hi:.4f}]")

# Show that NIfTI header spacing is read correctly (dz,dy,dx)
example_pred = next((ARTIFACTS_DIR / "data" / "seg").glob("*_pred.nii.gz"))
read_spacing = get_nifti_spacing(str(example_pred))
print(f"\nExample spacing from NIfTI header (as returned by get_nifti_spacing): {read_spacing}")
print("Expected spacing used when writing this demo (dz,dy,dx):", spacing)
if tuple(read_spacing) != tuple(spacing):
    print("NOTE: spacing axis order differs from the expected (dz,dy,dx) convention in this notebook.")
    print("      If you are using the PyPI build, upgrade to a medeval version that normalizes NIfTI axis/spacing order.")

# Per-case Dice histogram from CLI results.csv
import pandas as pd

seg_results = pd.read_csv(ARTIFACTS_DIR / "out_cli_seg" / "results.csv")
dice_vals = pd.to_numeric(seg_results.get("dice"), errors="coerce").dropna().to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dice_vals, bins=15)
ax.set_title("Segmentation Dice (per-case) — from CLI results.csv")
ax.set_xlabel("Dice")
ax.set_ylabel("#cases")
plt.tight_layout()
plt.show()

# ---------
# Classification / Detection / Registration CLI quick summary
# ---------
for name, out_dir, keys in [
    ("CLS", ARTIFACTS_DIR / "out_cli_cls", ["auroc", "auprc", "ece", "brier"]),
    ("DET", ARTIFACTS_DIR / "out_cli_det", ["mAP@0.50", "mAP@0.75", "precision", "recall"]),
    ("REG", ARTIFACTS_DIR / "out_cli_reg", ["nmi", "ncc"]),
]:
    s = json.loads((out_dir / "summary.json").read_text())
    print(f"\n{name} CLI processed: {s['n_processed']}/{s['n_samples']} (errors={s['n_errors']})")

    # Some package versions may not yet implement the full CLI for all tasks.
    if name == "DET" and "status_code" in s.get("metrics", {}):
        print("  NOTE: This medeval version does not yet report detection metrics in the CLI (placeholder only).")
        print("        The Detection capability is still demonstrated above via the Python API (compute_detection_metrics).")
        continue

    for k in keys:
        if k in s["metrics"]:
            print(f"  {k}: {s['metrics'][k]['mean']}")



In [ ]:
# torchmetrics-style wrapper (minimal stateful API)
from medeval.core.interop import MedEvalMetricWrapper
from medeval.metrics.segmentation import dice_score

wrapper = MedEvalMetricWrapper(dice_score, reduction="none")
wrapper.update(torch.ones(1, 10, 10), torch.ones(1, 10, 10))
wrapper.update(torch.zeros(1, 10, 10), torch.ones(1, 10, 10))
print(f"MedEvalMetricWrapper.compute(): {wrapper.compute().item():.4f} (mean over updates)")

# Optional: MONAI integration (only runs if monai is installed)
try:
    from medeval.core.interop import MONAITransformWrapper

    print("MONAI available: applying spacing transform wrapper to a toy dict...")
    data = {"image": torch.rand(1, 1, 8, 8, 8), "spacing": (2.0, 1.0, 1.0)}
    out = MONAITransformWrapper.apply_spacing_transform(data, keys=["image"], spacing_key="spacing")
    print(f"  input shape={tuple(data['image'].shape)} -> output shape={tuple(out['image'].shape)}")
except Exception as e:
    print(f"MONAI wrapper skipped: {e}")


In [ ]:
# Detection CLI was already run above in the self-contained end-to-end section.
# This cell just prints the stored summary for convenience.
import json
from pathlib import Path

ARTIFACTS_DIR = Path.cwd() / "medeval_demo_artifacts"
summary = json.loads((ARTIFACTS_DIR / "out_cli_det" / "summary.json").read_text())

if "status_code" in summary.get("metrics", {}):
    print("Detection CLI: placeholder only in this medeval version.")
    print("(Detection is demonstrated above via the Python API.)")
else:
    print("Detection CLI key metrics:")
    for k in ["mAP@0.50", "mAP@0.75", "precision", "recall"]:
        if k in summary["metrics"]:
            print(f"  {k}: {summary['metrics'][k]['mean']}")



## 10. PHI / Privacy Note

- **This notebook generates fully synthetic data** and writes it into `./medeval_demo_artifacts/`.
- MedEval is designed to evaluate predictions/targets and produce aggregate statistics and plots.
- If you use DICOM inputs, `medeval.core.io.load_dicom(..., strip_metadata=True)` loads pixel data while only keeping a small set of spacing-related tags (i.e., metadata stripping to avoid PHI leakage by default).
